In [8]:
import pandas as pd
import pymysql

In [5]:
conexion = pymysql.connect(
    host="dl-radar.cluster-ro-c7pmwdslewrp.us-east-1.rds.amazonaws.com",
    user = "debian",
    password= "eeAZU3v1FXCY9zmbvcS6kpEpyj",
    database="data_fact",
    port= 4408
)

cursor = conexion.cursor()

In [6]:
query = """
SELECT *
FROM base_rucs_sri;
"""
base_registro_civil = pd.read_sql_query(query, conexion)

C:\Users\anali\AppData\Local\Temp\ipykernel_29500\631102470.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  base_registro_civil = pd.read_sql_query(query, conexion)


In [27]:
base_registro_civil = pd.read_parquet(r"C:\Users\anali\OneDrive - PUBLIPROMUEVE S.A\Ruben Freire's files - CENTROS COMERCIALES\sandbox\bases\base_rucs_sri.parquet")

In [28]:
base_registro_civil['nombre_fantasia_comercial'] = (
    base_registro_civil['nombre_fantasia_comercial']
        .replace("", pd.NA)
)


In [29]:
base_registro_civil = base_registro_civil[base_registro_civil["nombre_fantasia_comercial"].notna()] 

In [30]:
base_registro_civil['motivo_cancelacion_suspension'] = (base_registro_civil['motivo_cancelacion_suspension'].replace("", pd.NA))

In [31]:
base_registro_civil = base_registro_civil[base_registro_civil['motivo_cancelacion_suspension'].isna()]

In [32]:
recreo_nombres = pd.read_excel(r"C:\Users\anali\OneDrive - PUBLIPROMUEVE S.A\Ruben Freire's files - CENTROS COMERCIALES\ccrecreo.xlsx")

In [33]:
dicc_nombre_fantasia = set(recreo_nombres['LOCAL'])

In [34]:
import unicodedata
import re

def normalizar(s):
    if pd.isna(s):
        return ""
    s = s.lower()
    s = unicodedata.normalize('NFD', s)
    s = ''.join(c for c in s if unicodedata.category(c) != 'Mn')
    s = re.sub(r'[^a-z0-9 ]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s


In [35]:
def qgrams(s, q=3):
    return {s[i:i+q] for i in range(len(s) - q + 1)}


In [36]:
def jaccard(a, b):
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)


In [37]:
dic_norm = {normalizar(x): x for x in dicc_nombre_fantasia}

dic_qgrams = {
    k: qgrams(k, q=3)
    for k in dic_norm.keys()
}


In [47]:
def match_qgram(nombre, threshold=0.7):
    s = normalizar(nombre)
    q_s = qgrams(s)

    mejor, score = None, 0
    for k, q_k in dic_qgrams.items():
        sim = jaccard(q_s, q_k)
        if sim > score:
            mejor, score = dic_norm[k], sim

    if score >= threshold:
        return mejor, score
    return None, score

In [48]:
base_registro_civil['nombre_fantasia_comercial'] = base_registro_civil['nombre_fantasia_comercial'].apply(normalizar) 

In [49]:
base_registro_civil_test = base_registro_civil[
    base_registro_civil['direccion_completa'].str.contains(
        r'(?=.*QUITO)(?=.*MARISCAL)',
        case=False,
        na=False,
        regex=True
    )
]


In [50]:
base_registro_civil_test['direccion_completa']

3564       PICHINCHA / QUITO / MARISCAL SUCRE / AV 12 DE ...
18844      PICHINCHA / QUITO / MARISCAL SUCRE / AV. DOCE ...
22425      PICHINCHA / QUITO / MARISCAL SUCRE / N21 B ROB...
22571      PICHINCHA / QUITO / MARISCAL SUCRE / JORGE WAS...
23571      PICHINCHA / QUITO / MARISCAL SUCRE / ERNESTO N...
                                 ...                        
7944601    PICHINCHA / QUITO / RUMIPAMBA / AV. MARISCAL S...
7944607    PICHINCHA / QUITO / MARISCAL SUCRE / CALLE VIC...
7944620    PICHINCHA / QUITO / MARISCAL SUCRE / AV. RIO A...
7944675    PICHINCHA / QUITO / LA CONCEPCIÓN / AV.MARISCA...
7944777    PICHINCHA / QUITO / MARISCAL SUCRE / AV. 12 DE...
Name: direccion_completa, Length: 16070, dtype: object

In [51]:
base_registro_civil_test[['match_qgram', 'score_qgram']] = (
    base_registro_civil_test['nombre_fantasia_comercial']
      .apply(lambda x: pd.Series(match_qgram(x)))
)

C:\Users\anali\AppData\Local\Temp\ipykernel_1632\404753551.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  base_registro_civil_test[['match_qgram', 'score_qgram']] = (
C:\Users\anali\AppData\Local\Temp\ipykernel_1632\404753551.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  base_registro_civil_test[['match_qgram', 'score_qgram']] = (


In [52]:
base_registro_civil_test = base_registro_civil_test.sort_values(by = 'score_qgram', ascending = False)

In [53]:
base_registro_civil_test[(base_registro_civil_test['match_qgram'].notna())][['razon_social', 'nombre_fantasia_comercial', 'match_qgram', 'score_qgram']]

,razon_social,nombre_fantasia_comercial,match_qgram,score_qgram
7942190,DISTRIBUIDORA FARMACEUTICA ECUATORIANA DIFARE ...,pharmacy s,PHARMACY’S,1.000000
7115441,CHAIDE Y CHAIDE S.A.,chaide,CHAIDE,1.000000
7115402,CHAIDE Y CHAIDE S.A.,chaide,CHAIDE,1.000000
6360304,ARIAS MITCHELL DIEGO JAVIER,diafoot,DIAFOOT,1.000000
6353772,MOSQUERA ARBOLEDA CARLOS SEBASTIAN,renova,RENOVA,1.000000
...,...,...,...,...
7114940,CYEDE CIA. LTDA.,digital photo express,KONICA DIGITAL PHOTO EXPRESS,0.730769
7114946,CYEDE CIA. LTDA.,digital photo express,KONICA DIGITAL PHOTO EXPRESS,0.730769
7194591,COOPERATIVA DE AHORRO Y CREDITO UNIVERSIDAD CA...,cooperativa de ahorro y credito p u c e,COOPERATIVA DE AHORRO Y CRÉDITO OSCUS,0.714286
6090830,CARRERA ESCOBAR ALEX EDUARDO,aece technology,BE TECHNOLOGY,0.714286


In [54]:
base_registro_civil_test[(base_registro_civil_test['nombre_fantasia_comercial'].str.contains("optica", case = False, na = False)) & ((base_registro_civil_test['match_qgram'].str.contains("óptica", case = False, na = False)))]

,id_establecimiento,numero_ruc,numero_establecimiento,razon_social,nombre_fantasia_comercial,cod_estado_contribuyente,estado_contribuyente,cod_estado_establecimiento,estado_establecimiento,matriz,...,fecha_cese_comercio,fecha_reinicio_actividades_comercio,fecha_actualizacion_comercio,nombre_representante_legal,identificacion_representante_legal,representantes_legales,fecha_actualizacion,encontrado,match_qgram,score_qgram
7136134,1.790559e+15,1.790559e+12,41.0,OPTICA LOS ANDES S.A.,optica los andes,1.0,ACTIVO,1.0,ABIERTO,0,...,None,None,2025-12-02,AGUILERA ORTIZ NATALIA LILIANA,1708730864,"[{""nombre"": ""AGUILERA ORTIZ NATALIA LILIANA"", ...",2026-01-09 19:57:08,1,ÓPTICA LOS ANDES,1.0
7136121,1.790559e+15,1.790559e+12,28.0,OPTICA LOS ANDES S.A.,optica los andes,1.0,ACTIVO,1.0,ABIERTO,0,...,None,None,2025-12-02,AGUILERA ORTIZ NATALIA LILIANA,1708730864,"[{""nombre"": ""AGUILERA ORTIZ NATALIA LILIANA"", ...",2026-01-09 19:57:08,1,ÓPTICA LOS ANDES,1.0
7136103,1.790559e+15,1.790559e+12,10.0,OPTICA LOS ANDES S.A.,optica los andes,1.0,ACTIVO,1.0,ABIERTO,0,...,None,None,2025-12-02,AGUILERA ORTIZ NATALIA LILIANA,1708730864,"[{""nombre"": ""AGUILERA ORTIZ NATALIA LILIANA"", ...",2026-01-09 19:57:08,1,ÓPTICA LOS ANDES,1.0
7136101,1.790559e+15,1.790559e+12,8.0,OPTICA LOS ANDES S.A.,optica los andes,1.0,ACTIVO,2.0,CERRADO,0,...,None,None,2025-12-02,AGUILERA ORTIZ NATALIA LILIANA,1708730864,"[{""nombre"": ""AGUILERA ORTIZ NATALIA LILIANA"", ...",2026-01-09 19:57:08,1,ÓPTICA LOS ANDES,1.0
7136099,1.790559e+15,1.790559e+12,6.0,OPTICA LOS ANDES S.A.,optica los andes,1.0,ACTIVO,2.0,CERRADO,0,...,None,None,2025-12-02,AGUILERA ORTIZ NATALIA LILIANA,1708730864,"[{""nombre"": ""AGUILERA ORTIZ NATALIA LILIANA"", ...",2026-01-09 19:57:08,1,ÓPTICA LOS ANDES,1.0
7136136,1.790559e+15,1.790559e+12,43.0,OPTICA LOS ANDES S.A.,optica los andes,1.0,ACTIVO,1.0,ABIERTO,0,...,None,None,2025-12-02,AGUILERA ORTIZ NATALIA LILIANA,1708730864,"[{""nombre"": ""AGUILERA ORTIZ NATALIA LILIANA"", ...",2026-01-09 19:57:08,1,ÓPTICA LOS ANDES,1.0
